# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one web page for one month. I will use March 2026 (month=2026-03) as my analysis window to develop and verify my content opportunity features. I will verify the actual row structure and date range using the warehouse data before finalizing this definition.

In [4]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

# --- SWAP THIS BLOCK for real data: df = pd.read_csv("your_file.csv", parse_dates=["date"]) ---
pairs = pd.DataFrame({"query_id": rng.integers(0, 80, 200),
                      "page_id": rng.integers(0, 30, 200)}).drop_duplicates()
days = pd.DataFrame({"date": pd.date_range("2025-01-01", periods=90)})
df = pairs.merge(days, how="cross").sample(frac=0.4, random_state=42).reset_index(drop=True)
df["impressions"] = rng.poisson(40, len(df)) + 1
df["position"] = rng.uniform(1, 30, len(df)).round(1)
df["clicks"] = rng.binomial(df["impressions"], np.clip(0.35 / df["position"], 0, 1))
df["ctr"] = df["clicks"] / df["impressions"]
df["source"] = np.where(df["date"] < "2025-02-01", "gsc_only", "gsc+analytics")
df["sessions"] = np.where(df["source"] == "gsc_only", np.nan,
                          df["clicks"] * rng.uniform(0.8, 1.1, len(df)))
# ---------------------------------------------------------------------------------------------

KEY = ["query_id", "page_id", "date"]          # claim: one row = one query-page-day
print("rows:", len(df))
print("duplicate keys:", df.duplicated(KEY).sum(), "(0 means the grain holds)")
print("window:", df["date"].min().date(), "->", df["date"].max().date())
print("distinct days:", df["date"].nunique())

rows: 6984
duplicate keys: 0 (0 means the grain holds)
window: 2025-01-01 -> 2025-03-31
distinct days: 90


## 2. Fields: feature / label / context / excluded

### Features
- `avg_position`: Average search position of a page.
- `ctr`: Click-through rate of a page.
- `clicks`: Number of clicks.
- `impressions`: Number of times a page appeared in search results.

### Label
- Content opportunity score or proxy, to be defined after inspecting the available data.

### Context
- `page`: Identifies the web page.
- `month`: Identifies the reporting month.

### Excluded
- Future-month performance: excluded because it would not be available when making the decision.
- Any field derived directly from the label: excluded from the final features to prevent data leakage.

In [5]:
FIELDS = {
    "feature":  ["impressions", "position"],
    "label":    ["clicks"],
    "context":  ["query_id", "page_id", "date", "source"],
    "excluded": {
        "ctr":      "derived from the label (clicks / impressions), so it leaks",
        "sessions": "different system, missing in early rows, near-copy of clicks",
    },
}

assigned = (FIELDS["feature"] + FIELDS["label"] + FIELDS["context"]
            + list(FIELDS["excluded"]))
print("unassigned columns:", sorted(set(df.columns) - set(assigned)))
print("listed but not in data:", sorted(set(assigned) - set(df.columns)))

# leakage check: how closely does each excluded/feature field track the label?
for c in ["ctr", "sessions", "impressions", "position"]:
    print(f"corr({c}, clicks) = {df[c].corr(df['clicks']):.2f}")

unassigned columns: []
listed but not in data: []
corr(ctr, clicks) = 0.98
corr(sessions, clicks) = 0.99
corr(impressions, clicks) = 0.10
corr(position, clicks) = -0.59


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
print("shape:", df.shape)
print("\nnull rate per column:")
print(df.isna().mean().round(3).to_string())

print("\nrows per week:")
print(df.groupby(df["date"].dt.to_period("W")).size().describe().round(1).to_string())

print("\nrows per source:")
print(df["source"].value_counts().to_string())

print("\nlabel (clicks) distribution:")
print(df["clicks"].describe().round(2).to_string())
print("share of rows with 0 clicks:", round((df["clicks"] == 0).mean(), 3))

print("\nrange sanity:")
print("position min/max:", df["position"].min(), df["position"].max())
print("clicks > impressions:", (df["clicks"] > df["impressions"]).sum())

shape: (6984, 9)

null rate per column:
query_id       0.000
page_id        0.000
date           0.000
impressions    0.000
position       0.000
clicks         0.000
ctr            0.000
source         0.000
sessions       0.345

rows per week:
count     14.0
mean     498.9
std      128.3
min       77.0
25%      520.8
50%      540.0
75%      548.2
max      571.0

rows per source:
source
gsc+analytics    4573
gsc_only         2411

label (clicks) distribution:
count    6984.00
mean        1.71
std         2.47
min         0.00
25%         0.00
50%         1.00
75%         2.00
max        22.00
share of rows with 0 clicks: 0.349

range sanity:
position min/max: 1.0 30.0
clicks > impressions: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
# 1) Concentration: do a few pairs dominate?
by_pair = df.groupby(["query_id", "page_id"])["clicks"].sum().sort_values(ascending=False)
top10 = by_pair.head(max(1, int(len(by_pair) * 0.10)))
print("top 10% of pairs hold", round(top10.sum() / by_pair.sum(), 3), "of all clicks")

# 2) Early rows: which fields are missing before the second source starts?
print("\nnull rate by source:")
print(df.groupby("source")[["sessions"]].apply(lambda g: g.isna().mean()).round(3).to_string())

# 3) Time-split overlap: same pairs on both sides of a cutoff = leakage risk
cut = pd.Timestamp("2025-02-15")
train, test = df[df["date"] < cut], df[df["date"] >= cut]
pair = lambda d: set(zip(d["query_id"], d["page_id"]))
shared = pair(train) & pair(test)
print(f"\ntrain rows {len(train)}, test rows {len(test)}")
print("pairs seen in both:", len(shared), "of", len(pair(test)), "test pairs")

# 4) Coverage: share of the calendar each pair is actually observed
span = df["date"].nunique()
cov = df.groupby(["query_id", "page_id"])["date"].nunique() / span
print("median days observed per pair:", round(cov.median(), 3), "of the window")

top 10% of pairs hold 0.159 of all clicks

null rate by source:
               sessions
source                 
gsc+analytics       0.0
gsc_only            1.0

train rows 3479, test rows 3505
pairs seen in both: 194 of 194 test pairs
median days observed per pair: 0.4 of the window


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.